# E-commerce Sales Analysis

Exploratory analysis of the sales dataset (`projetoportfolio.bronze.*`) combining data from sales, products, customers, and competitor prices. This notebook demonstrates core Python data analysis skills — from basic data structures and API integration to Spark-to-pandas workflows and business KPI generation — answering key questions: Which sales channel drives the most revenue? How are top products and customers distributed? What growth opportunities exist?


## Parameters

In [0]:
nome_tabela = "vendas"
total_vendas = 3020
receita_total = 974077.28
pipeline_ativo = True

for valor in (nome_tabela, total_vendas, receita_total, pipeline_ativo):
    print(valor, type(valor))

In [0]:
print(f"A tabela {nome_tabela} tem {total_vendas} linhas.")

# Formatação BRL: separador de milhar com ponto e 2 casas decimais
print(f"Receita: R$ {receita_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

## Data Structures

In [0]:
tabelas = ["vendas", "produtos", "clientes", "preco_competidores"]

print(f"São {len(tabelas)} tabelas")
print("Primeira:", tabelas[0])
print("Última:  ", tabelas[-1])     # índice negativo conta de trás para frente

In [0]:
venda = {
    "id_venda": "sal_adff6978b0c6",
    "canal": "loja_fisica",
    "quantidade": 2,
    "preco_unitario": 64.79,
}

receita = venda["quantidade"] * venda["preco_unitario"]
print(f"Receita da venda {venda['id_venda']}: R$ {receita:.2f}")

In [0]:
estado = {"sigla": "AM", "nome": "Amazonas", "regiao": {"id": 1, "sigla": "N", "nome": "Norte"}}

print(estado["regiao"]["nome"])

In [0]:
tabelas = ["vendas", "produtos", "clientes", "preco_competidores"]

for tabela in tabelas:
    print(f"Baixando {tabela}.parquet do data lake...")

In [0]:
quantidades = [1, 3, 2, 5, 1]

for quantidade in quantidades:
    if quantidade > 2:
        print(f"{quantidade} unidades: venda grande")
    else:
        print(f"{quantidade} unidades: venda pequena")

In [0]:
def calcular_receita(quantidade, preco_unitario):
    """Receita de uma venda, arredondada em 2 casas."""
    return round(quantidade * preco_unitario, 2)


print(calcular_receita(2, 64.79))
print(calcular_receita(1, 1286.09))

## External Data & Pandas

In [0]:
import requests
import pandas as pd

print("pandas", pd.__version__)
print("requests", requests.__version__)

In [0]:
URL_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"

resposta = requests.get(URL_IBGE, timeout=60)
resposta.raise_for_status()

estados = resposta.json()

print("status:", resposta.status_code)
print("estados recebidos:", len(estados))
print("primeiro:", estados[0])

In [0]:
# Extração de sigla, nome e região de cada estado
linhas = [{"sigla": e["sigla"], "nome": e["nome"], "regiao": e["regiao"]["nome"]} for e in estados]

df_estados = pd.DataFrame(linhas)
display(df_estados.head())

In [0]:
print("formato (linhas, colunas):", df_estados.shape)
print("colunas:", list(df_estados.columns))

por_regiao = df_estados.groupby("regiao").size().sort_values(ascending=False)
print(por_regiao)

display(df_estados[df_estados["regiao"] == "Norte"])

## Data Lake Integration

In [0]:
tabela_vendas = "projetoportfolio.bronze.vendas"

# Ler com Spark (retorna um Spark DataFrame)
df_vendas_spark = spark.table(tabela_vendas)

print(f"Tabela: {tabela_vendas}")
print(f"Linhas: {df_vendas_spark.count():,}")
print(f"Colunas: {len(df_vendas_spark.columns)}")
print(f"\nEsquema:")
df_vendas_spark.printSchema()

In [0]:
# Mostrar as primeiras 5 linhas
print("Primeiras 5 vendas:")
df_vendas_spark.show(5)

# Consulta SQL simples: receita total
receita_total = df_vendas_spark.selectExpr("SUM(quantidade * preco_unitario) as receita_total").collect()[0][0]
print(f"\nReceita total: R$ {receita_total:,.2f}")

In [0]:
# Atenção: converter apenas amostras pequenas (Spark é distribuído, pandas é local)

df_vendas_pandas = df_vendas_spark.limit(100).toPandas()

print(f"Tipo original: {type(df_vendas_spark)}")
print(f"Tipo convertido: {type(df_vendas_pandas)}")
print(f"\nForma: {df_vendas_pandas.shape}")
print(f"Colunas: {list(df_vendas_pandas.columns)}")

print(f"\nReceita média: R$ {df_vendas_pandas['preco_unitario'].mean():.2f}")
display(df_vendas_pandas.head())

## Exploratory Analysis

In [0]:
# Carregando as 4 tabelas do projeto portfolio
tabelas = {
    "vendas": spark.table("projetoportfolio.bronze.vendas").limit(1000).toPandas(),
    "produtos": spark.table("projetoportfolio.bronze.produtos").toPandas(),
    "clientes": spark.table("projetoportfolio.bronze.clientes").toPandas(),
    "competidores": spark.table("projetoportfolio.bronze.preco_competidores").toPandas(),
}

print("Datasets carregados:")
for nome, df in tabelas.items():
    print(f"  {nome:15} {df.shape[0]:>4} linhas x {df.shape[1]:>2} colunas")

In [0]:
# Mapeamento produto_id → nome e cliente_id → estado
produtos_dict = dict(zip(tabelas["produtos"]["id_produto"], tabelas["produtos"]["nome_produto"]))
clientes_dict = dict(zip(tabelas["clientes"]["id_cliente"], tabelas["clientes"]["estado"]))

print(f"Criados {len(produtos_dict)} produtos e {len(clientes_dict)} clientes indexados")
print(f"\nExemplo - Produto prd_96fbc500aec1: {produtos_dict.get('prd_96fbc500aec1', 'N/A')}")

In [0]:
df_vendas = tabelas["vendas"].copy()

df_vendas["receita"] = df_vendas.apply(
    lambda row: calcular_receita(row["quantidade"], row["preco_unitario"]), 
    axis=1
)

df_vendas["produto"] = df_vendas["id_produto"].map(produtos_dict)
df_vendas["estado"] = df_vendas["id_cliente"].map(clientes_dict)

print("Dataset enriquecido:")
print(df_vendas[["id_venda", "produto", "estado", "quantidade", "preco_unitario", "receita"]].head())

In [0]:
receita_total = df_vendas["receita"].sum()
total_vendas = len(df_vendas)
ticket_medio = receita_total / total_vendas
clientes_unicos = df_vendas["id_cliente"].nunique()

print("=" * 50)
print("KPIs PRINCIPAIS")
print("=" * 50)
print(f"Receita Total:      R$ {receita_total:>12,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
print(f"Total de Vendas:       {total_vendas:>12,}".replace(",", "."))
print(f"Ticket Médio:       R$ {ticket_medio:>12,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
print(f"Clientes Únicos:      {clientes_unicos:>12,}".replace(",", "."))
print("=" * 50)

In [0]:
top_produtos = (
    df_vendas.groupby("produto")["receita"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

print("\nTOP 5 PRODUTOS POR RECEITA:")
for i, (produto, receita) in enumerate(top_produtos.items(), 1):
    produto_nome = produto if produto else "Sem categoria"
    print(f"{i}. {produto_nome[:40]:<40} R$ {receita:>10,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

In [0]:
por_canal = df_vendas.groupby("canal_venda").agg({
    "receita": "sum",
    "id_venda": "count"
}).rename(columns={"id_venda": "vendas"})

por_canal["ticket_medio"] = por_canal["receita"] / por_canal["vendas"]

print("\nANÁLISE POR CANAL:")
for canal in por_canal.index:
    receita = por_canal.loc[canal, "receita"]
    vendas = por_canal.loc[canal, "vendas"]
    ticket = por_canal.loc[canal, "ticket_medio"]
    participacao = (receita / receita_total) * 100
    
    print(f"\n{canal.upper()}:")
    print(f"  Receita: R$ {receita:,.2f} ({participacao:.1f}%)".replace(",", "X").replace(".", ",").replace("X", "."))
    print(f"  Vendas:  {int(vendas):,}".replace(",", "."))
    print(f"  Ticket:  R$ {ticket:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

## Results Visualization

In [0]:
import matplotlib.pyplot as plt

# Gráfico de barras: receita por canal
fig, ax = plt.subplots(figsize=(10, 6))

por_canal["receita"].plot(kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e"])
ax.set_title("Receita por Canal de Venda", fontsize=16, fontweight="bold")
ax.set_xlabel("Canal", fontsize=12)
ax.set_ylabel("Receita (R$)", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Adicionar valores nas barras
for i, v in enumerate(por_canal["receita"]):
    ax.text(i, v + 5000, f"R$ {v:,.0f}".replace(",", "."), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

In [0]:
# Gráfico horizontal: top produtos
fig, ax = plt.subplots(figsize=(10, 6))

top_produtos.plot(kind="barh", ax=ax, color="#2ca02c")
ax.set_title("Top 5 Produtos por Receita", fontsize=16, fontweight="bold")
ax.set_xlabel("Receita (R$)", fontsize=12)
ax.set_ylabel("")

# Adicionar valores nas barras
for i, v in enumerate(top_produtos.values):
    ax.text(v + 500, i, f"R$ {v:,.0f}".replace(",", "."), va="center", fontweight="bold")

plt.tight_layout()
plt.show()

## Business Insights

In [0]:
# Gerando insights automáticos com base nos dados
print("=" * 70)
print("INSIGHTS DE NEGÓCIO")
print("=" * 70)

# Insight 1: Canal dominante
canal_top = por_canal["receita"].idxmax()
participacao_top = (por_canal.loc[canal_top, "receita"] / receita_total) * 100
print(f"\n1. CANAL DOMINANTE")
print(f"   O canal '{canal_top}' representa {participacao_top:.1f}% da receita total.")
print(f"   → Recomendação: Investir em marketing digital para fortalecer esse canal.")

# Insight 2: Ticket médio por canal
canal_maior_ticket = por_canal["ticket_medio"].idxmax()
canal_menor_ticket = por_canal["ticket_medio"].idxmin()
print(f"\n2. DIFERENÇA DE TICKET MÉDIO")
print(f"   {canal_maior_ticket}: R$ {por_canal.loc[canal_maior_ticket, 'ticket_medio']:.2f}".replace(".", ","))
print(f"   {canal_menor_ticket}: R$ {por_canal.loc[canal_menor_ticket, 'ticket_medio']:.2f}".replace(".", ","))
diferenca_pct = ((por_canal.loc[canal_maior_ticket, 'ticket_medio'] / por_canal.loc[canal_menor_ticket, 'ticket_medio']) - 1) * 100
print(f"   → O ticket de {canal_maior_ticket} é {diferenca_pct:.1f}% maior que {canal_menor_ticket}.")

# Insight 3: Concentração de receita
top_5_receita = top_produtos.sum()
concentracao = (top_5_receita / receita_total) * 100
print(f"\n3. CONCENTRAÇÃO DE RECEITA")
print(f"   Os top 5 produtos representam {concentracao:.1f}% da receita total.")
if concentracao > 50:
    print(f"   Alta concentração! Risco caso algum produto tenha problema de estoque.")
    print(f"   → Recomendação: Diversificar portfólio e monitorar estoque dos top produtos.")
else:
    print(f"   Portfólio diversificado, menor risco de concentração.")

# Insight 4: Base de clientes
ticket_medio_formatted = f"R$ {ticket_medio:.2f}".replace(".", ",")
print(f"\n4. OPORTUNIDADE DE CRESCIMENTO")
print(f"   {clientes_unicos} clientes únicos geraram {total_vendas} vendas.")
frequencia = total_vendas / clientes_unicos
print(f"   Frequência média: {frequencia:.2f} vendas por cliente.")
if frequencia < 2:
    print(f"   → Oportunidade: Implementar programa de fidelidade para aumentar recorrência.")

print("\n" + "=" * 70)

### Conclusion

**Key Findings:**
1. **E-commerce dominates** with 75% of revenue → focus on digital strategy
2. **15% higher average ticket in e-commerce** → online customers spend more per transaction
3. **Balanced portfolio** → top 5 products = 38% of revenue (low concentration risk)
4. **High recurrence** → 20 sales per customer on average


**Dashboard**:

[![Management Dashboard](https://img.shields.io/badge/Management_Dashboard-Open-blue?style=for-the-badge)](https://dbc-7f9e15f9-4844.cloud.databricks.com/dashboardsv3/01f1bb542faa1f77a6cdac3f773f8cb1/published?o=7474644802236927)